# 05 — H1-N results ledger and limitations

This notebook reads saved artifacts rather than transcribing values. It deliberately separates D0 legacy diagnostics from H1-N controlled results. Empty H1-N tables mean that no relevant controlled experiment has been completed; they do not mean a metric is zero.

In [ ]:
from pathlib import Path
import json

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL

artifact_root = Path('../artifacts')

def h1n_results(root: Path) -> pd.DataFrame:
    rows = []
    for run_path in root.glob('*/run.json'):
        metrics_path = run_path.parent / 'internal_test_metrics.json'
        if not metrics_path.is_file():
            continue
        run = json.loads(run_path.read_text(encoding='utf-8'))
        if run.get('preprocessing', {}).get('protocol') != CONTROLLED_PREPROCESSING_PROTOCOL:
            continue
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        config = run.get('config', {})
        rows.append({
            'status': 'exploratory_internal_stress_test',
            'run': run_path.parent.name,
            'representation': config.get('representation'),
            'seed': config.get('seed'),
            'preprocessing': run['preprocessing']['protocol'],
            'image_size': run['preprocessing']['image_size'],
            'threshold': run.get('threshold'),
            **metrics,
        })
    return pd.DataFrame(rows)

results = h1n_results(artifact_root)
if results.empty:
    print('PENDING: no completed H1-N internal result is available.')
else:
    display(results.sort_values(['representation', 'seed']))

## D0 diagnostic ledger — not a result table for H1-N

These artifacts document the geometry/source confound and are retained for reproducibility. They are excluded from all controlled comparisons, seed aggregates, external-validation claims, model selection, and interface decisions.

In [ ]:
d0_rows = []
for run_name in ('radial_logistic_seed7', 'file_metadata_control_seed7'):
    metrics_path = artifact_root / run_name / 'internal_test_metrics.json'
    if metrics_path.is_file():
        d0_rows.append({
            'status': 'D0 diagnostic only — not H1-N evidence',
            'run': run_name,
            **json.loads(metrics_path.read_text(encoding='utf-8')),
        })

d0_results = pd.DataFrame(d0_rows)
if d0_results.empty:
    print('No D0 metric artifact is present.')
else:
    display(d0_results)

## Three-seed exploratory aggregation

Only after all three predeclared seeds have completed for both representations may the internal stress-test rows be aggregated. The aggregation is descriptive, not confirmatory: D0 already opened the same Defactify test split. `fpr_at_tpr_95` is likewise a descriptive value from each test-set ROC curve, not the validation-selected operating threshold recorded in `run.json`. Preserve per-seed rows, per-generator tables and cluster-bootstrap comparison artifacts beside this summary.

In [ ]:
EXPECTED_SEEDS = {7, 17, 42}
if not results.empty and set(results.representation) == {'rgb', 'fft'} and all(
    set(results.loc[results.representation == representation, 'seed']) == EXPECTED_SEEDS
    for representation in ('rgb', 'fft')
):
    aggregate = (
        results.groupby('representation')[['roc_auc', 'pr_auc', 'balanced_accuracy', 'macro_f1', 'fpr_at_tpr_95']]
        .agg(['mean', 'std'])
        .round(4)
    )
    display(aggregate)
else:
    print('PENDING: aggregate only after RGB and FFT each contain seeds 7, 17 and 42.')

## Confirmatory external and robustness ledger

The external Synthbuster + RAISE evaluation remains locked until the H1-N plan is frozen. Its generator-specific, macro and worst-generator metrics must be kept separate from Defactify internal stress-test values and stratified by the explicit prepared-manifest relation: same-named generator, same-family version, or unseen family. Thus it is inaccurate to call all Synthbuster rows unseen. Robustness rows are valid only if the evaluator has been checked to apply the same H1-N centre-crop and 128 × 128 raster before each fixed degradation.

In [ ]:
external_rows = []
for prediction_path in artifact_root.glob('*/external_predictions.csv'):
    external_rows.append({'run': prediction_path.parent.name, 'path': str(prediction_path)})

if external_rows:
    display(pd.DataFrame(external_rows))
else:
    print('PENDING: no external prediction artifact exists; this is expected before the lock is released.')

## Limitations and permitted wording

- Geometry/source confounding required a protocol amendment; D0 must not be promoted to detector performance.
- H1-N is a controlled Defactify representation comparison, not universal image authentication.
- The original internal test is exploratory because it was seen during D0; confirmatory evidence requires the locked external corpus.
- FFT magnitude discards colour and phase and may still exploit corpus- or generator-specific artefacts.
- Model scores are not probabilities until a separately documented calibration procedure has been completed.

Use measured wording only: *On the specified exploratory internal stress test, [model] achieved [metric] under the H1-N protocol.* Only after the frozen external run may the report add: *On the separate locked Synthbuster + RAISE evaluation, the frozen model achieved [metric].* Never claim that the score proves an arbitrary image's origin.